# se-dat walkthrough — Titanic dataset

`se-dat` (Simple Exploratory Data Analysis) takes a raw DataFrame and gives you:

1. **Column type profiling** — inferred semantic types, missing %, cardinality, confidence flags.
2. **Correlation analysis** — Pearson/Spearman (numeric), Cramér's V (categorical), correlation ratio η (mixed), heatmaps, multicollinearity flags.
3. **Encoding suggestions** — binary maps for `yes/no` style columns, one-hot for low-cardinality categoricals, target/ordinal encoding for high-cardinality ones.

Available on PyPI: [`pip install se-dat`](https://pypi.org/project/se-dat/)

This notebook runs the whole pipeline on the classic Titanic passenger data.

In [ ]:
# If you haven't installed the package yet, uncomment:
# %pip install se-dat

## 1. Load the data

In [ ]:
import pandas as pd
import sedat

print("se-dat version:", sedat.__version__)

URL = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(URL)
df.head()

In [ ]:
print(f"{df.shape[0]} rows x {df.shape[1]} columns")
df.describe(include="all").T

What we already know going in:

- `Survived` (0/1) is our prediction target.
- `Age` has missing values, `Cabin` is mostly missing.
- `Sex` and `Embarked` are strings, `Pclass` is a number but really an ordinal category,
- `Name`, `Ticket`, `PassengerId` are high-cardinality text/ids.

Let's see how much of that `se-dat` recovers automatically.

## 2. One-call report: `EDAReport.create`

The facade runs profiling, correlations and encoding suggestions in one shot.
Passing `target=` enables target-encoding suggestions for high-cardinality categoricals.

In [ ]:
report = sedat.EDAReport.create(df, target=df["Survived"])

In [ ]:
report.profile.summary

Things to notice in the profile:

- `PassengerId` was recognized as an **id** column (unique integers).
- `Survived` (int 0/1) is flagged **boolean** with *medium* confidence — 0/1 is ambiguous between a flag and a real number.
- `Age` shows its **missing_pct**, `Cabin` shows ~77% missing.
- `Sex`, `Embarked` land in **categorical**; `Name`/`Ticket` in **string** (high-cardinality free text).

In [ ]:
report.correlations.flagged_pairs

In [ ]:
report.encoding_summary

## 3. Profiling, piece by piece

Everything in the report is also available as standalone functions.

In [ ]:
profile = sedat.profile_dataframe(df)
profile.summary

In [ ]:
# Inspect inference on individual, tricky columns
for col in ["PassengerId", "Survived", "Cabin"]:
    print(f"--- {col} ---")
    print(sedat.infer_column_type(df[col]), end="\n\n")

## 4. Correlation analysis

`se-dat` covers three pairings:

| pairing | measure | function |
|---|---|---|
| numeric ↔ numeric | Pearson / Spearman | `numeric_correlation` |
| categorical ↔ categorical | Cramér's V | `categorical_correlation` |
| numeric ↔ categorical | correlation ratio η | `numeric_categorical_correlation` |

In [ ]:
pearson = sedat.numeric_correlation(df, method="pearson")
pearson

In [ ]:
fig = sedat.correlation_heatmap(pearson, title="Pearson correlation (numeric columns)");

### Spearman (rank-based)

`Pclass` vs `Fare` is stronger under Spearman than Pearson — the fare distribution is heavily skewed, so ranks tell a cleaner story.

In [ ]:
spearman = sedat.numeric_correlation(df, method="spearman")
fig = sedat.correlation_heatmap(spearman, title="Spearman correlation (rank-based)");

### Cramér's V (categorical ↔ categorical)

> Caveat: Cramér's V is biased upward for high-cardinality columns, so take the `Ticket`/`Cabin` numbers with a grain of salt — `Sex` vs `Embarked` is the meaningful cell here.

In [ ]:
cramers = sedat.categorical_correlation(df)
fig = sedat.correlation_heatmap(cramers, title="Cramér's V (categorical columns)");

In [ ]:
# Correlation ratio: how well does each categorical column explain each numeric column?
eta = sedat.numeric_categorical_correlation(df)
eta.round(3)

In [ ]:
fig = sedat.correlation_heatmap(eta, title="Correlation ratio η (rows: numeric, cols: categorical)");

In [ ]:
# Pairs above the |0.7| threshold — potential multicollinearity
sedat.correlation_report(df, threshold=0.7).flagged_pairs

## 5. Encoding suggestions

Strategies chosen per column:

| situation | strategy |
|---|---|
| binary-like strings (`yes/no`) | `binary_encode` → 0/1 |
| categorical with ≤ `cardinality_threshold` values | `one_hot` |
| high-cardinality categorical **with** target | `target_encode` |
| high-cardinality categorical **without** target | `ordinal_encode` (warns) |

On Titanic that means: `Sex` → binary map, `Embarked` → one-hot. Everything else is left alone.

In [ ]:
plan = sedat.suggest_encodings(df, target=df["Survived"], cardinality_threshold=10)
plan.summary

### Applying a single suggestion

You don't have to accept everything — pick suggestions individually:

In [ ]:
sex_step = next(s for s in plan.suggestions if s.column == "Sex")
step_df = sex_step.apply(df)

step_df[["Sex", "Embarked"]].head()

### Applying everything at once

`apply_all_encodings` returns a new frame — the original `df` is never mutated.

In [ ]:
model_ready = report.apply_all_encodings()
model_ready.dtypes.to_frame("dtype")

In [ ]:
model_ready.head()

## 6. Bonus: feed it straight into a model

Sanity check that the transformed frame is genuinely model-ready (needs `scikit-learn`).

In [ ]:
try:
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import cross_val_score
    from sklearn.pipeline import make_pipeline
except ImportError:
    print("scikit-learn not installed — skipping this cell (%pip install scikit-learn to run it).")
else:
    X = (
        model_ready
        .drop(columns=["Survived"])
        .select_dtypes("number")
        .drop(columns=["PassengerId"])
    )
    y = model_ready["Survived"]

    pipe = make_pipeline(SimpleImputer(strategy="median"), LogisticRegression(max_iter=1000))
    scores = cross_val_score(pipe, X, y, cv=5)
    print(f"LogisticRegression 5-fold CV accuracy: {scores.mean():.3f} ± {scores.std():.3f}")

## Wrap-up

The full loop in four lines:

```python
import sedat

report = sedat.EDAReport.create(df, target=df["Survived"])
report.profile.summary          # what did I load?
report.correlations.flagged_pairs  # what's redundant?
model_ready = report.apply_all_encodings()  # make it trainable
```

See the README for the complete API (`profile_column`, `cramers_v`, `correlation_ratio`, ...).